In [13]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import dagshub

In [14]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
780,I just discovered this film and love it. Just ...,positive
954,I first saw this in the theater in 1969 when I...,positive
729,This is one horror movie based TV show that ge...,positive
93,"Just saw it....the story, the plot, the script...",negative
178,It's nice to see Julie Andrews trying a straig...,negative


In [15]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\SUNIL KUMAR\AppData\Local\Temp\ipykernel_52628\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [16]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

df = normalize_text(df)
df.head()
df.head()

[nltk_data] Downloading package stopwords to C:\Users\SUNIL
[nltk_data]     KUMAR\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\SUNIL
[nltk_data]     KUMAR\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\SUNIL
[nltk_data]     KUMAR\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,review,sentiment
780,discovered film love it right mix fast moving ...,positive
954,first saw theater immediately fell love it sad...,positive
729,one horror movie based tv show get right frida...,positive
93,saw it the story plot script make absolute sen...,negative
178,nice see julie andrew trying straight dramatic...,negative


In [17]:
df['sentiment'].value_counts()

sentiment
positive    259
negative    241
Name: count, dtype: int64

In [18]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [19]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
780,discovered film love it right mix fast moving ...,1
954,first saw theater immediately fell love it sad...,1
729,one horror movie based tv show get right frida...,1
93,saw it the story plot script make absolute sen...,0
178,nice see julie andrew trying straight dramatic...,0


In [20]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [21]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [23]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/vanshatajaiswal4/MLOPS.mlflow')
dagshub.init(repo_owner='vanshatajaiswal4', repo_name='MLOPS', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-04-27 17:40:32,856 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/vanshatajaiswal4/MLOPS "HTTP/1.1 200 OK"


Initialized MLflow to track repo "vanshatajaiswal4/MLOPS"

2026-04-27 17:40:32,879 - INFO - Initialized MLflow to track repo "vanshatajaiswal4/MLOPS"


Repository vanshatajaiswal4/MLOPS initialized!

2026-04-27 17:40:32,880 - INFO - Repository vanshatajaiswal4/MLOPS initialized!


<Experiment: artifact_location='mlflow-artifacts:/7242b1e84475448d8479be741cdfe898', creation_time=1777287823351, experiment_id='0', last_update_time=1777287823351, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [24]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-04-27 17:40:35,694 - INFO - Starting MLflow run...
2026-04-27 17:40:36,242 - INFO - Logging preprocessing parameters...
2026-04-27 17:40:37,353 - INFO - Initializing Logistic Regression model...
2026-04-27 17:40:37,359 - INFO - Fitting the model...
2026-04-27 17:40:37,429 - INFO - Model training complete.
2026-04-27 17:40:37,430 - INFO - Logging model parameters...
2026-04-27 17:40:37,789 - INFO - Making predictions...
2026-04-27 17:40:37,795 - INFO - Calculating evaluation metrics...
2026-04-27 17:40:37,816 - INFO - Logging evaluation metrics...
2026-04-27 17:40:39,295 - INFO - Saving and logging the model...
2026/04/27 17:40:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:40:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run silent-hawk-4 at: https://dagshub.com/vanshatajaiswal4/MLOPS.mlflow/#/experiments/0/runs/cb30e8c9f7d149289fec9e396b9b4e90
🧪 View experiment at: https://dagshub.com/vanshatajaiswal4/MLOPS.mlflow/#/experiments/0
